# M06-02 — Cache y particionado

Referencia de validación. El alumno trabaja en `notebooks/alumno/M06-02-cache-particionado.ipynb`.


## Celda 0 — localizar el repo


In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


In [ ]:
from pyspark.sql.functions import col, lit
spark = get_spark("novashop-m06")
base = spark.read.parquet(str(STAGING / "fact_lines"))
xl = base.withColumn("_copy", lit(-1))
for i in range(7):
    xl = xl.unionByName(base.withColumn("_copy", lit(i)))
print("particiones iniciales", xl.rdd.getNumPartitions())
assert xl.count() == 1980 * 8
import time
def timed_count(df, label):
    t0 = time.perf_counter()
    n = df.where(col("is_billable")).count()
    print(label, n, f"{time.perf_counter() - t0:.2f}s")
    return n
assert timed_count(xl, "1er count frío") == 9016
warm = xl.where(col("is_billable")).cache()
assert timed_count(warm, "calentamiento") == 9016
assert timed_count(warm, "caliente") == 9016
by_month = warm.repartition(12, col("order_month"))
print("particiones", by_month.rdd.getNumPartitions())
assert by_month.rdd.getNumPartitions() == 12
by_month.groupBy("order_month").count().orderBy("order_month").show()
warm.unpersist()
print("M06-02 OK")
